In [1]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        """
        root_dir = path to 'real-vs-fake' folder
        split = 'train' or 'valid'
        """
        self.samples = []
        self.transform = transform

        # 🔥 Kaggle structure fix
        base_path = os.path.join(root_dir, split)

        for label, cls in enumerate(["real", "fake"]):
            cls_path = os.path.join(base_path, cls)

            if not os.path.exists(cls_path):
                print(f"Warning: {cls_path} not found")
                continue

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
                    img_path = os.path.join(cls_path, img_name)
                    self.samples.append((img_path, label))

        print(f"{split.upper()} → {len(self.samples)} samples loaded")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")
        img = np.array(img)

        if self.transform:
            img = self.transform(image=img)["image"]

        return img, torch.tensor(label, dtype=torch.float32)

In [2]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_transforms(train=True):
    if train:
        return A.Compose([
            A.Resize(224, 224),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=[0.485,0.456,0.406],
                        std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=[0.485,0.456,0.406],
                        std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])

In [3]:
from torch.utils.data import DataLoader

def get_dataloaders(data_path, batch_size=32):
    train_ds = DeepfakeDataset(data_path, split="train",
                              transform=get_transforms(True))
    
    val_ds = DeepfakeDataset(data_path, split="valid",
                            transform=get_transforms(False))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [4]:
def generate_views(img_bgr):
    """Generate multiple views of the image for ensemble prediction"""
    # Convert to RGB if needed
    if len(img_bgr.shape) == 3 and img_bgr.shape[2] == 3:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2RGB) if len(img_bgr.shape) == 2 else img_bgr
    
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # Edge detection
    edges = cv2.Canny(gray, 100, 200)
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

    # FFT magnitude spectrum
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    magnitude = 20 * np.log(np.abs(fshift) + 1)
    magnitude = np.clip(magnitude, 0, 255).astype(np.uint8)
    magnitude = cv2.cvtColor(magnitude, cv2.COLOR_GRAY2RGB)

    # Noise residual
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    noise = cv2.absdiff(gray, blur)
    noise = (noise / noise.max() * 255).astype(np.uint8)
    noise = cv2.cvtColor(noise, cv2.COLOR_GRAY2RGB)

    return [img_rgb, edges, magnitude, noise]

In [5]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

def preprocess_image(img_rgb, image_size=224):
    transform = A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    augmented = transform(image=img_rgb)
    return augmented['image'].unsqueeze(0)

In [6]:
import numpy as np
def fuzzify(p):
    low = max(0, 1 - 2 * p)
    medium = 1 - abs(p - 0.5) * 2
    high = max(0, 2 * (p - 0.5))
    return low, medium, high


def fuzzy_rules(px, pe):
    lx, mx, hx = fuzzify(px)
    le, me, he = fuzzify(pe)

    fake = min(hx, he)
    real = min(lx, le)
    uncertain = max(mx, me)

    return fake, real, uncertain


def weighted_mean(results, weights):
    fake = sum(w * r[0] for w, r in zip(weights, results))
    real = sum(w * r[1] for w, r in zip(weights, results))
    uncertain = sum(w * r[2] for w, r in zip(weights, results))

    return fake, real, uncertain


def weighted_defuzzify(fake, real, uncertain, threshold=0.6):
    score = (fake * 1 + uncertain * 0.6 + real * 0) / (1 + 0.6)
    label = "FAKE" if score > threshold else "REAL"
    return label, score

In [7]:
import torch
import torch.nn as nn
import timm

class DeepfakeClassifier(nn.Module):
    def __init__(self, model_name="efficientnet_b0", num_classes=1, dropout=0.2):
        super().__init__()
        
        # disable pretrained (Kaggle internet issue)
        self.model = timm.create_model(model_name, pretrained=False, drop_rate=dropout)

        #  handle both EfficientNet and Xception
        if "efficientnet" in model_name:
            in_features = self.model.classifier.in_features
            self.model.classifier = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_features, num_classes)
            )
        else:
            in_features = self.model.get_classifier().in_features
            self.model.reset_classifier(num_classes)

    def forward(self, x):
        return self.model(x)


def get_model(model_name="efficientnet_b0", device="cuda"):
    model = DeepfakeClassifier(model_name)
    return model.to(device)


def predict(model, img_tensor, device="cuda"):
    model.eval()
    with torch.no_grad():
        img_tensor = img_tensor.to(device)
        out = model(img_tensor)
        prob = torch.sigmoid(out).item()
    return prob

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), correct / total


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), correct / total


def train_model(model, train_loader, val_loader, epochs=5, lr=1e-4, device="cuda"):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print("-"*40)

In [9]:
import cv2
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2



def run_inference(img_path_or_array, model_x, model_e, device="cuda", threshold=0.6):

    if isinstance(img_path_or_array, str):
        img = cv2.imread(img_path_or_array)
        if img is None:
            raise ValueError(f"Could not load image from {img_path_or_array}")
    else:
        img = img_path_or_array
    
    views = generate_views(img)
    results = []

    view_weights = [0.4, 0.2, 0.2, 0.2]

    for view in views:
        tensor = preprocess_image(view)

        px = predict(model_x, tensor, device)
        pe = predict(model_e, tensor, device)

        results.append(fuzzy_rules(px, pe))

    fake, real, uncertain = weighted_mean(results, view_weights)
    label, score = weighted_defuzzify(fake, real, uncertain, threshold)

    confidence = 1.0 - uncertain
    return label, score, confidence

In [10]:
import os
import torch

def run_main(image_path):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Loading models...")
    model_x = get_model("xception41", device)
    model_e = get_model("efficientnet_b0", device)

    # (optional) load trained weights if you saved them
    if os.path.exists("xception.pth"):
        model_x.load_state_dict(torch.load("xception.pth", map_location=device))

    if os.path.exists("efficientnet.pth"):
        model_e.load_state_dict(torch.load("efficientnet.pth", map_location=device))

    print("Running inference...")
    label, score, confidence = run_inference(image_path, model_x, model_e, device)

    print("\n" + "="*50)
    print(f"PREDICTION: {label}")
    print(f"FAKE Score: {score:.4f}")
    print(f"Confidence: {confidence:.4f}")
    print("="*50)

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Create models
model_x = get_model("xception41", device)
model_e = get_model("efficientnet_b0", device)

Using device: cuda


In [12]:
DATA_PATH = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"

train_loader, val_loader = get_dataloaders(DATA_PATH, batch_size=32)

print("Train size:", len(train_loader.dataset))
print("Val size:", len(val_loader.dataset))

TRAIN → 100000 samples loaded
VALID → 20000 samples loaded
Train size: 100000
Val size: 20000


In [13]:
# Train Xception
print("Training Xception...")
train_model(model_x, train_loader, val_loader, epochs=30, device=device)

# Train EfficientNet
print("Training EfficientNet...")
train_model(model_e, train_loader, val_loader, epochs=30, device=device)

Training Xception...


KeyboardInterrupt: 

In [ ]:
torch.save(model_x.state_dict(), "/kaggle/working/xception.pth")
torch.save(model_e.state_dict(), "/kaggle/working/efficientnet.pth")

print("Models saved successfully")

In [ ]:
# Pick some validation samples and test
for i in range(5):
    path, label = val_loader.dataset.samples[i]

    pred, score, conf = run_inference(path, model_x, model_e, device)

    print("Image:", path)
    print("Actual:", "FAKE" if label == 1 else "REAL")
    print("Predicted:", pred)
    print("Score:", round(score, 4))
    print("Confidence:", round(conf, 4))
    print("-"*50)

In [ ]:
import numpy as np

# -------------------------------
# MODEL EVALUATION (Single Model)
# -------------------------------
def evaluate_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.numpy()

            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds = (outputs > 0.5).astype(int).flatten()

            all_preds.extend(preds)
            all_labels.extend(labels)

            correct += (preds == labels).sum()
            total += len(labels)

    accuracy = correct / total
    return accuracy, np.array(all_preds), np.array(all_labels)


# -------------------------------
# FUZZY SYSTEM EVALUATION
# -------------------------------
def evaluate_fuzzy(loader, model_x, model_e, device, n=1000):
    correct = 0

    for i in range(min(n, len(loader.dataset.samples))):
        path, label = loader.dataset.samples[i]

        pred, _, _ = run_inference(path, model_x, model_e, device)
        pred = 1 if pred == "FAKE" else 0

        if pred == label:
            correct += 1

    return correct / n


# -------------------------------
# RUN EVALUATION
# -------------------------------
print("Evaluating models...")

acc_x, preds_x, labels = evaluate_model(model_x, val_loader, device)
acc_e, preds_e, _ = evaluate_model(model_e, val_loader, device)
acc_f = evaluate_fuzzy(val_loader, model_x, model_e, device)

print("\n" + "="*50)
print(f"Xception Accuracy      : {acc_x:.4f}")
print(f"EfficientNet Accuracy  : {acc_e:.4f}")
print(f"Fuzzy System Accuracy  : {acc_f:.4f}")
print("="*50)

In [ ]:
train_model(model_e, train_loader, val_loader, epochs=5, device=device)

In [ ]:
acc_e, _, _ = evaluate_model(model_e, val_loader, device)
print("EfficientNet Accuracy:", acc_e)

In [ ]:
import matplotlib.pyplot as plt

models = ["Xception", "EfficientNet", "Fuzzy System"]
accuracies = [acc_x, acc_e, acc_f]

plt.figure(figsize=(6,4))
plt.bar(models, accuracies)
plt.ylim(0, 1)
plt.title("Model Comparison (Deepfake Detection)")
plt.ylabel("Accuracy")
plt.xlabel("Models")

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center')

plt.show()